# ETL Transform: Stocks

This notebook runs the **stocks ETL pipeline**: ingest from Postgres (with warmup window) → transform (returns, volatility, technical indicators) → save to `historical_processed` → publish to S3.

**S3 upload modes (set in config cell below):**
- **Per day**: one CSV per book per day at `stocks/transformed/crypto/book={book}/year=.../month=.../day=.../format=csv/YYYYMMDD-{book}.csv`
- **Batch (run/week/month/year)**: one CSV per run (all books) or per (book, partition) at `stocks/transformed/crypto/book={book}/year=.../week=...` or `month=...` or `year=.../format=csv/...`

Set `AWS_STOCKS_BUCKET` or `AWS_DEFAULT_BUCKET` in `.env` for S3 uploads.

In [3]:
import sys
from pathlib import Path

# Resolve project root: run from repo root or notebooks/etl/
_cwd = Path(".").resolve()
project_root = _cwd if (_cwd / "src").is_dir() else (_cwd.parent.parent if _cwd.name == "etl" else _cwd)
src_path = project_root / "src"
if src_path.is_dir():
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(src_path))
else:
    raise FileNotFoundError(f"Expected src at {src_path}. Run from repo root or notebooks/etl/.")

import pandas as pd

In [ ]:
# Config: date range, books, and S3 upload options
SINCE = "2025-01-01"
UNTIL = "2025-12-31"
BOOKS = ["btc-usd"]  # None = all books; or e.g. ["btc-usd", "eth-usd"]
WARMUP_DAYS = 252
# S3: per-day (one file per book per day) and/or batch (one file per book per week/month/year)
UPLOAD_S3 = True
UPLOAD_S3_BATCH = ["year"]  # e.g. ["run", "week", "month", "year"] or None

In [4]:
# Run stocks ETL: transform (with warmup for indicators), save to Postgres, publish to S3.

import logging
# Show pipeline progress in the notebook (ingest, transform, save, S3 upload)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S", force=True)
for _name in ("pipelines.etl_transform", "pipelines.etl_cli", "transform.stocks.stock_transformers"):
    logging.getLogger(_name).setLevel(logging.INFO)

from pipelines.etl_transform import run_stocks_etl

transformed_df = run_stocks_etl(
    since=SINCE,
    until=UNTIL,
    books=BOOKS,
    warmup_days=WARMUP_DAYS,
    stocks_bucket=None,  # uses AWS_STOCKS_BUCKET or AWS_DEFAULT_BUCKET from .env
    save_to_postgres=False,
    upload_s3=UPLOAD_S3,
    upload_s3_batch=UPLOAD_S3_BATCH,
)

print(f"Transformed {len(transformed_df)} stock records")

16:54:11 - INFO - Ingesting stocks (warmup 2024-04-24 to 2025-12-31)...


16:54:11 - INFO - ============================================================
16:54:12 - INFO - INGEST STOCKS
16:54:12 - INFO - ============================================================
16:54:12 - INFO - Fetching stocks from database...
16:54:12 - INFO -   Books: ['btc-usd']
16:54:12 - INFO - Retrieved 5311 records
16:54:12 - INFO - Filtered since 2024-04-24: 891 records
16:54:12 - INFO - Filtered until 2025-12-31: 741 records
16:54:12 - INFO - Ingestion complete: 741 records
16:54:12 - INFO - Transforming (returns, volatility, technical indicators)...
16:54:12 - INFO - Stock transformation pipeline initialized
16:54:12 - INFO - Transforming 741 stock records...
16:54:12 - INFO - Stock transformation complete: 741 records
16:54:12 - INFO - Transformed 463 stock records (2025-01-01 to 2025-12-31)


Connection to the database successful!
Table name set to: historical
Connection closed.


16:54:12 - INFO - Uploading 364 book/day files to s3://test-financial-stocks-bucket/...
16:55:32 - INFO - Uploaded 364 group files to s3://test-financial-stocks-bucket/


Transformed 463 stock records


In [5]:
# Inspect transformed output
if not transformed_df.empty:
    display(transformed_df.head())
    print(transformed_df.columns.tolist())

,ref,book,date,open,high,low,close,adj_close,volume,simple_return,...,sma_200,ema_12,ema_26,rsi_14,macd,macd_signal,macd_histogram,bb_upper,bb_middle,bb_lower
278,https://finance.yahoo.com,btc-usd,2025-01-01,93425.10,94929.87,92788.13,94419.76,94419.76,24519888919,0.010602,...,73442.49735,95305.531656,96351.232356,42.467681,-1045.700700,-400.317067,-645.383633,106151.892271,97619.1095,89086.326729
279,https://finance.yahoo.com,btc-usd,2025-01-02,94416.29,97739.82,94201.57,96886.88,96886.88,46009564411,0.026129,...,73619.68980,95548.816016,96390.909959,48.076377,-842.093943,-488.672442,-353.421501,105501.917578,97323.2125,89144.507422
280,https://finance.yahoo.com,btc-usd,2025-01-03,96881.73,98956.91,96034.62,98107.43,98107.43,35611391163,0.012598,...,73795.71970,95942.448937,96518.059592,51.895977,-575.610655,-506.060085,-69.550570,104521.879707,97013.6490,89505.418293
281,https://finance.yahoo.com,btc-usd,2025-01-04,98106.99,98734.43,97562.98,98236.23,98236.23,22342608078,0.001313,...,73972.39360,96295.338331,96645.331474,57.354849,-349.993143,-474.846696,124.853554,102863.787306,96623.9745,90384.161694
282,https://finance.yahoo.com,btc-usd,2025-01-05,98233.91,98813.30,97291.77,98314.96,98314.96,20525254825,0.000801,...,74156.20445,96606.049357,96769.007661,58.661554,-162.958304,-412.469018,249.510714,100685.337895,96232.6925,91780.047105


['ref', 'book', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'simple_return', 'log_return', 'volatility_20d', 'volatility_60d', 'volatility_parkinson', 'volatility_gk', 'sma_20', 'sma_50', 'sma_200', 'ema_12', 'ema_26', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'bb_upper', 'bb_middle', 'bb_lower']


## Optional: Specific books and date range

In [6]:
# transformed_df = run_stocks_etl(
#     since="2026-01-01",
#     until="2026-01-28",
#     books=["btc-usd", "eth-usd"],
#     warmup_days=252,
#     save_to_postgres=True,
#     upload_s3=True,
# )